# Text Cleaning and Preprocessing — LMSYS ToxicChat

**Owner:** Hadia (`Hadia-hj`)  
**Dataset:** `lmsys/toxic-chat`, configuration `toxicchat0124`  
**Text field:** `user_input`  
**Target:** `toxicity` (0 = non-toxic, 1 = toxic)

## Purpose and reproducibility

This notebook inspects real user prompts, compares Unicode and case-handling options, evaluates conservative noise handling, demonstrates stemming and lemmatization, and builds one reusable cleaning pipeline. Original text is retained separately from cleaned text.

**Important:** Do not upload raw dataset files or generated CSV outputs to GitHub. The notebook downloads the public dataset when run. Run every cell from top to bottom before submission.

In [ ]:
# Install dependencies in Colab/Jupyter if they are not already installed
%pip -q install datasets pandas nltk matplotlib

import re
import unicodedata
from pathlib import Path

import pandas as pd
import nltk
from datasets import load_dataset
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import TreebankWordTokenizer

# POS tagging and WordNet data are needed for the final lemmatization step.
for resource in ["averaged_perceptron_tagger_eng", "wordnet", "omw-1.4"]:
    nltk.download(resource, quiet=True)

print("Imports and NLTK resources are ready.")

In [ ]:
DATASET_NAME = "lmsys/toxic-chat"
DATASET_CONFIG = "toxicchat0124"
TEXT_COLUMN = "user_input"
LABEL_COLUMN = "toxicity"

dataset = load_dataset(DATASET_NAME, DATASET_CONFIG)

train_df = dataset["train"].to_pandas().copy()
test_df = dataset["test"].to_pandas().copy()

required_columns = {TEXT_COLUMN, LABEL_COLUMN}
for split_name, frame in [("train", train_df), ("test", test_df)]:
    missing = required_columns.difference(frame.columns)
    if missing:
        raise ValueError(f"{split_name} split is missing columns: {sorted(missing)}")

print("Train shape:", train_df.shape)
print("Test shape: ", test_df.shape)
display(train_df[[TEXT_COLUMN, LABEL_COLUMN]].head())

## 1. Inspect examples before cleaning

The examples below are selected from the actual training split. The search looks for Unicode characters, URLs, contractions, repeated punctuation, or unusual whitespace. Review the output and note that meaningful words, negation, capitalization, numbers, and expressive punctuation can carry toxicity-related information.

In [ ]:
messy_pattern = r"[^\x00-\x7F]|https?://|www\.|n't|['’]|[!?]{2,}|\s{2,}"
messy_mask = train_df[TEXT_COLUMN].fillna("").astype(str).str.contains(
    messy_pattern, regex=True
)
messy_examples = train_df.loc[messy_mask, [TEXT_COLUMN, LABEL_COLUMN]].head(5)

if messy_examples.empty:
    messy_examples = train_df[[TEXT_COLUMN, LABEL_COLUMN]].head(5)

for index, row in messy_examples.iterrows():
    print(f"Row index: {index} | toxicity: {row[LABEL_COLUMN]}")
    print(repr(row[TEXT_COLUMN]))
    print("-" * 80)

**Observation and preservation decision:** User prompts may contain Unicode variants, contractions, URLs, repeated punctuation, mixed capitalization, and irregular spacing. I preserve the message's words, negation, capitalization, numbers, and punctuation because these can express intent, emphasis, or sentiment. URLs and user mentions are replaced with neutral placeholders so that their variable contents do not dominate the vocabulary. This is a conservative choice for toxicity classification.

## 2. Unicode normalization

NFC combines canonically equivalent Unicode sequences while usually preserving compatibility distinctions. NFKC additionally folds compatibility characters (for example, full-width letters and some presentation forms). Compare both on actual dataset messages and on a small illustrative example.

In [ ]:
unicode_examples = train_df[TEXT_COLUMN].dropna().astype(str).head(5)

for text in unicode_examples:
    print("Original:", repr(text))
    print("NFC:     ", repr(unicodedata.normalize("NFC", text)))
    print("NFKC:    ", repr(unicodedata.normalize("NFKC", text)))
    print("-" * 80)

compatibility_example = "ＡＢＣ ① ﬁ"
print("Compatibility example:", repr(compatibility_example))
print("NFC: ", unicodedata.normalize("NFC", compatibility_example))
print("NFKC:", unicodedata.normalize("NFKC", compatibility_example))

**Decision — use NFC.** NFC standardizes canonically equivalent character sequences without applying the broader compatibility conversions performed by NFKC. This is a conservative choice for user-generated text because unusual presentation forms may be intentional. The dataset is predominantly English; no Urdu/Arabic-specific character folding is applied.

## 3. Case handling

Compare preserving case, lowercasing, and Unicode casefolding. Capitalization can be a useful signal: all-caps words may indicate shouting or emphasis, and proper names may be relevant to context.

In [ ]:
case_examples = train_df[TEXT_COLUMN].dropna().astype(str).head(5)

for text in case_examples:
    print("Original: ", repr(text))
    print("Lowercase:", repr(text.lower()))
    print("Casefold: ", repr(text.casefold()))
    print("-" * 80)

**Decision — preserve capitalization.** The final pipeline does not lowercase or casefold the complete message. Uppercase emphasis and named entities can be useful features for toxicity detection. During lemmatization only, each token is temporarily lowercased for WordNet lookup and its original capitalization pattern is restored.

## 4. Noise handling and stopword decision

The pipeline uses conservative replacements:
- URLs become `URL`.
- User mentions such as `@name` become `USER`.
- Repeated whitespace is collapsed.
- Punctuation, numbers, hashtags, contractions, and negation are retained.

Stopwords are **not removed**. Removing common words can change sentence meaning, especially when negation words such as `not`, `no`, and `never` are removed.

In [ ]:
def replace_noise(text):
    """Replace variable URL/mention strings and normalize whitespace."""
    text = str(text)
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text, flags=re.IGNORECASE)
    text = re.sub(r"(?<!\w)@\w+", " USER ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

noise_examples = train_df[TEXT_COLUMN].dropna().astype(str).head(8)
for text in noise_examples:
    print("Before:", repr(text))
    print("After: ", repr(replace_noise(text)))
    print("-" * 80)

# Explicitly verify that negation is not removed.
negation_examples = ["I am not happy.", "No, I never said that."]
for text in negation_examples:
    print(repr(text), "->", repr(replace_noise(text)))

**Decision:** No stopword-removal list is applied. Negation changes the meaning of a message and may reverse whether it expresses harmful intent. Punctuation and numbers are also retained because they can convey emphasis or context. URLs and mentions are replaced rather than deleted, preserving the fact that such content was present.

## 5. Stemming and lemmatization

Stemming uses heuristic suffix removal and can create non-dictionary forms. Lemmatization uses vocabulary and part-of-speech information to produce a dictionary form. The assignment asks that one method be selected for the final pipeline, so this notebook uses **POS-aware WordNet lemmatization** and does not apply stemming to the final text.

In [ ]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

demo_words = [
    ("running", "v"),
    ("studies", "n"),
    ("cars", "n"),
    ("better", "a"),
    ("playing", "v"),
]

for word, wn_pos in demo_words:
    print(
        f"{word:12} | stem: {stemmer.stem(word):12} | "
        f"lemma: {lemmatizer.lemmatize(word, pos=wn_pos)}"
    )

# Demonstrate POS-aware lemmatization on a short sentence.
demo_sentence = "The children were running and playing with better toys."
tokenizer = TreebankWordTokenizer()
demo_tokens = tokenizer.tokenize(demo_sentence)
tagged_tokens = nltk.pos_tag(demo_tokens)

def penn_to_wordnet(tag):
    if tag.startswith("J"):
        return "a"
    if tag.startswith("V"):
        return "v"
    if tag.startswith("N"):
        return "n"
    if tag.startswith("R"):
        return "r"
    return None

demo_lemmas = []
for token, tag in tagged_tokens:
    wn_pos = penn_to_wordnet(tag)
    lemma = lemmatizer.lemmatize(token.lower(), pos=wn_pos) if wn_pos else token
    demo_lemmas.append(lemma)

print("Sentence:", demo_sentence)
print("Tokens:  ", demo_tokens)
print("POS tags:", tagged_tokens)
print("Lemmas:  ", demo_lemmas)

**Decision — apply lemmatization, not stemming.** Lemmatization generally produces interpretable dictionary forms and avoids some of the truncated outputs created by stemming. POS-aware lemmatization is selected to meet the assignment requirement while keeping the transformation relatively conservative. A limitation is that WordNet may not recognize slang, misspellings, or abusive terms; those tokens are therefore left unchanged when no reliable WordNet transformation applies. Capitalization is restored after lookup.

## 6. Build and validate the final cleaning pipeline

The order is: (1) null handling, (2) NFC normalization, (3) URL/mention replacement, (4) tokenization and POS-aware lemmatization, and (5) whitespace/punctuation reconstruction. The same deterministic function is applied to train and test. No vocabulary, stopword list, or other learned preprocessing parameter is fitted on the test split.

In [ ]:
TOKEN_PATTERN = re.compile(r"\w+(?:['’]\w+)*|[^\w\s]", flags=re.UNICODE)
wordnet_lemmatizer = WordNetLemmatizer()

def penn_to_wordnet(tag):
    """Map Penn Treebank POS tags to WordNet POS tags."""
    if tag.startswith("J"):
        return "a"
    if tag.startswith("V"):
        return "v"
    if tag.startswith("N"):
        return "n"
    if tag.startswith("R"):
        return "r"
    return None

def restore_case(original, lemma):
    """Restore all-caps or initial-capital style after lowercase WordNet lookup."""
    if original.isupper():
        return lemma.upper()
    if original[:1].isupper():
        return lemma[:1].upper() + lemma[1:]
    return lemma

def clean_text(text):
    """Clean one ToxicChat prompt while preserving its original meaning signals."""
    if pd.isna(text):
        return ""

    text = unicodedata.normalize("NFC", str(text))
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text, flags=re.IGNORECASE)
    text = re.sub(r"(?<!\w)@\w+", " USER ", text)

    tokens = TOKEN_PATTERN.findall(text)
    if not tokens:
        return ""

    tagged_tokens = nltk.pos_tag(tokens)
    cleaned_tokens = []

    for token, tag in tagged_tokens:
        wn_pos = penn_to_wordnet(tag)
        if wn_pos and re.search(r"\w", token, flags=re.UNICODE):
            lemma = wordnet_lemmatizer.lemmatize(token.lower(), pos=wn_pos)
            token = restore_case(token, lemma)
        cleaned_tokens.append(token)

    # Rejoin tokens and attach punctuation to the preceding token.
    cleaned = " ".join(cleaned_tokens)
    cleaned = re.sub(r"\s+([,.;:!?%])", r"\1", cleaned)
    cleaned = re.sub(r"([([{])\s+", r"\1", cleaned)
    cleaned = re.sub(r"\s+([)\]}])", r"\1", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned

# Preserve original text exactly (including nulls in the original source column).
for frame in (train_df, test_df):
    frame["original_text"] = frame[TEXT_COLUMN]
    frame["cleaned_text"] = frame["original_text"].apply(clean_text)

print("Train null cleaned values:", train_df["cleaned_text"].isna().sum())
print("Train empty cleaned values:", (train_df["cleaned_text"].str.strip() == "").sum())
print("Test null cleaned values: ", test_df["cleaned_text"].isna().sum())
print("Test empty cleaned values: ", (test_df["cleaned_text"].str.strip() == "").sum())

display(train_df[["original_text", "cleaned_text", LABEL_COLUMN]].head(10))

## 7. Before/after comparison, vocabulary, and decisions

The examples below are taken from the processed training split. Vocabulary is measured as the number of unique whitespace-independent word tokens using a consistent Unicode-aware regular expression. The values are computed from the actual dataset at runtime; do not replace them with guessed numbers.

In [ ]:
comparison = train_df[["original_text", "cleaned_text", LABEL_COLUMN]].head(5)
display(comparison)

def get_word_tokens(series):
    tokens = []
    for value in series.fillna("").astype(str):
        tokens.extend(re.findall(r"\b\w+\b", value.casefold(), flags=re.UNICODE))
    return tokens

original_tokens = get_word_tokens(train_df["original_text"])
cleaned_tokens = get_word_tokens(train_df["cleaned_text"])

vocab_before = len(set(original_tokens))
vocab_after = len(set(cleaned_tokens))

print("Training documents:", len(train_df))
print("Testing documents: ", len(test_df))
print("Vocabulary size before cleaning:", vocab_before)
print("Vocabulary size after cleaning: ", vocab_after)
print("Token count before cleaning:", len(original_tokens))
print("Token count after cleaning: ", len(cleaned_tokens))
print("Vocabulary reduction (%):",
      round((1 - vocab_after / vocab_before) * 100, 2) if vocab_before else 0)

print("\nToxicity distribution (train):")
display(train_df[LABEL_COLUMN].value_counts(dropna=False).rename_axis("toxicity").to_frame("count"))

In [ ]:
decisions = pd.DataFrame({
    "Step": [
        "Unicode normalization",
        "Case handling",
        "URLs",
        "User mentions",
        "Punctuation and numbers",
        "Stopwords / negation",
        "Morphology"
    ],
    "Decision": [
        "NFC",
        "Preserve original capitalization",
        "Replace with URL",
        "Replace with USER",
        "Preserve",
        "Do not remove stopwords; preserve negation",
        "Apply POS-aware lemmatization; no stemming"
    ],
    "Reason / risk": [
        "Canonical consistency with limited compatibility changes",
        "Uppercase can signal emphasis; proper names retain form",
        "Avoids variable URL strings while retaining their presence",
        "Avoids user-specific handles dominating vocabulary",
        "May carry emotion or context",
        "Negation changes sentence meaning",
        "Dictionary forms are interpretable; slang may remain unchanged"
    ]
})
display(decisions)

## 8. Save cleaned outputs for the next project phase

The following files are written locally under `data/processed/`. They are not committed to GitHub because the repository instructions say not to upload raw dataset files; the team should decide whether generated processed outputs are stored externally or tracked under its agreed data policy. Each CSV contains the original message, cleaned message, and toxicity label.

In [ ]:
output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

columns_to_save = ["original_text", "cleaned_text", LABEL_COLUMN]
train_output = output_dir / "toxicchat_train_cleaned.csv"
test_output = output_dir / "toxicchat_test_cleaned.csv"

train_df[columns_to_save].to_csv(train_output, index=False)
test_df[columns_to_save].to_csv(test_output, index=False)

print("Saved:", train_output.resolve())
print("Saved:", test_output.resolve())
print("Train output shape:", pd.read_csv(train_output).shape)
print("Test output shape: ", pd.read_csv(test_output).shape)

## 9. References and limitations

1. LMSYS, *ToxicChat: Unveiling Hidden Challenges of Toxicity Detection in Real-World User-AI Conversation*. Dataset card: https://huggingface.co/datasets/lmsys/toxic-chat
2. Dataset configuration: `toxicchat0124`; text field: `user_input`; target: `toxicity`.
3. Dataset license: **CC-BY-NC-4.0**. This is a non-commercial license; follow the dataset card's terms and attribution requirements.
4. Paper: Lin et al. (2023), *ToxicChat: Unveiling Hidden Challenges of Toxicity Detection in Real-World User-AI Conversation*, arXiv:2310.17389.

**Limitations:** WordNet lemmatization is English-oriented and may not recognize internet slang, misspellings, or novel abusive terms. The cleaning function deliberately does not delete punctuation, numbers, stopwords, or negation. Any final model should be evaluated with and without selected transformations to check whether the choices improve downstream performance.

**Submission check:** Restart the runtime and run every cell from top to bottom. Confirm both cleaned CSVs are generated locally, inspect the before/after examples and vocabulary counts, and capture screenshots of successful execution for the assignment report.